# 02 — Persiapan Anotasi NER

Notebook ini dipakai **setelah notebook 01 selesai**. Tujuannya memilih cuplikan artikel yang layak ditinjau, membaginya kepada tiga anggota, dan membuat template anotasi manual.

**Penting:** notebook ini tidak mengisi label entitas secara otomatis. Label final harus ditinjau manusia. Jangan gunakan test set untuk tuning model.

## Pembagian kerja

- Anggota 1: anotasi antrean 1 dan pemeriksaan struktur dataset.
- Anggota 2: anotasi antrean 2 dan persiapan notebook training (tahap berikutnya).
- Anggota 3: anotasi antrean 3 dan mulai rancangan API/web (tahap berikutnya).
- Setelah selesai, satukan hasil, lakukan pemeriksaan silang, lalu baru tetapkan split train/validation/test berdasarkan article ID. Pembagian anggota bukan pembagian train/test.

In [ ]:
import io, json, zipfile, re, random, shutil
from pathlib import Path
import pandas as pd
from google.colab import files

SEED = 42
OUT = Path('/content/ner_annotation_pack')
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
print('Siap. Unggah NER_Profiling_Results.zip dari notebook 01.')

## 1. Unggah hasil profiling

Pilih ZIP yang dibuat notebook 01, yaitu `NER_Profiling_Results.zip`. Jika notebook 01 belum selesai, kembali dulu dan jalankan sampai akhir.

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise ValueError('Unggah NER_Profiling_Results.zip terlebih dahulu.')
zip_name = next((n for n in uploaded if n.lower().endswith('.zip')), None)
if not zip_name:
    raise ValueError('File yang diunggah bukan ZIP.')

with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as zf:
    names = zf.namelist()
    print('File di dalam ZIP:', names)
    csv_name = next((n for n in names if n.endswith('articles_normalized_for_review.csv')), None)
    if not csv_name:
        raise FileNotFoundError('articles_normalized_for_review.csv tidak ditemukan. Pastikan ZIP berasal dari notebook 01.')
    articles = pd.read_csv(io.BytesIO(zf.read(csv_name)), dtype=str, keep_default_na=True)
    conflict_name = next((n for n in names if n.endswith('conflicting_article_ids.csv')), None)
    if conflict_name:
        conflicts = pd.read_csv(io.BytesIO(zf.read(conflict_name)), dtype=str, keep_default_na=True)
    else:
        conflicts = pd.DataFrame()

articles.columns = [str(c).strip().lower() for c in articles.columns]
print(f'Artikel yang dibaca: {len(articles):,}')
print('Kolom:', articles.columns.tolist())
display(articles.head(3))

## 2. Audit konflik ID dan pilih bahasa

Agar kebocoran data tidak terjadi, artikel yang memiliki konflik metadata untuk `original_id` perlu diperiksa dan tidak otomatis masuk antrean anotasi. Untuk baseline pertama, kita fokus pada artikel berbahasa Inggris (`en`) karena jumlahnya paling banyak di profil sebelumnya.

In [ ]:
required = ['original_id', 'title', 'body', 'language']
missing = [c for c in required if c not in articles.columns]
if missing:
    raise ValueError(f'Kolom wajib tidak ditemukan: {missing}')

# Daftar ID konflik bila ada
conflict_ids = set()
if not conflicts.empty and 'original_id' in conflicts.columns:
    conflict_ids = set(conflicts['original_id'].dropna().astype(str).str.strip())
print('ID konflik dari audit profiling:', len(conflict_ids))

# Normalisasi ringan untuk pemilihan; simpan teks sumber untuk review
for c in ['original_id', 'title', 'body', 'language']:
    articles[c] = articles[c].astype('string').str.strip()

work = articles.copy()
work = work[work['original_id'].notna() & work['original_id'].ne('')]
work = work[~work['original_id'].isin(conflict_ids)]
work = work[work['language'].fillna('').str.lower().str.strip().eq('en')]
work = work[work['title'].notna() & work['title'].ne('') & work['body'].notna() & work['body'].ne('')]
work = work.drop_duplicates(subset=['original_id'], keep='first').copy()
work['body_char_count'] = work['body'].astype(str).str.len()
print(f'Artikel bahasa Inggris yang lolos filter dasar: {len(work):,}')
print('Distribusi kandidat panjang isi:')
display(work['body_char_count'].describe(percentiles=[.25,.5,.75,.9]).to_frame())

## 3. Buat cuplikan pendek yang mudah dianotasi

Tugas mengizinkan artikel pendek atau cuplikan artikel untuk set evaluasi. Di sini kita buat cuplikan dari maksimal tiga kalimat pertama dan batasi panjangnya agar anotasi manusia lebih terkelola. Hasil ini tetap mempertahankan `original_id` dan referensi artikel. Periksa cuplikan sebelum memberi label; perbaiki bila pemotongan membuat teks tidak masuk akal.

In [ ]:
def make_excerpt(text, max_chars=1200, max_sentences=3):
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if not text:
        return ''
    # Pemisahan kalimat sederhana; hasil perlu ditinjau manusia.
    parts = re.split(r'(?<=[.!?])\s+', text)
    excerpt = ' '.join(parts[:max_sentences]).strip()
    if len(excerpt) > max_chars:
        excerpt = excerpt[:max_chars].rsplit(' ', 1)[0].rstrip(' ,;:-')
    return excerpt

work['body_excerpt'] = work['body'].map(make_excerpt)
work['excerpt_char_count'] = work['body_excerpt'].str.len()
work = work[work['excerpt_char_count'] >= 80].copy()
print('Kandidat setelah membuat cuplikan:', len(work))
display(work[['original_id','title','language','body_excerpt','excerpt_char_count']].head(5))

## 4. Pilih antrean awal untuk tiga anggota

Target awal rekomendasi: **120 cuplikan**, masing-masing 40 per anggota. Ini adalah target kerja yang realistis untuk kelompok tiga orang, bukan angka minimum resmi dari soal. Jika jumlah kandidat kurang dari 120, notebook memakai semua kandidat yang tersedia.

Teks anotasi akan berbentuk `judul + baris baru + cuplikan isi`. Semua posisi `start` dan `end` harus dihitung terhadap kolom `text_to_annotate` secara persis.

In [ ]:
TARGET = 120
if len(work) == 0:
    raise ValueError('Tidak ada kandidat. Periksa language, isi CSV, dan hasil profiling.')

n_take = min(TARGET, len(work))
selected = work.sample(n=n_take, random_state=SEED).copy().reset_index(drop=True)
selected['text_to_annotate'] = selected['title'].fillna('').astype(str).str.strip() + '\n' + selected['body_excerpt'].fillna('').astype(str).str.strip()
selected['annotations_json'] = ''  # sengaja kosong: isi manual setelah membaca teks
selected['annotator'] = ''
selected['review_notes'] = ''
selected['article_ref'] = selected['original_id']

keep = ['article_ref','original_id','title','language','body_excerpt','text_to_annotate','annotations_json','annotator','review_notes']
selected = selected[keep].copy()
selected.to_csv(OUT/'all_annotation_items.csv', index=False)

# Acak urutan kerja, lalu bagi agar tiap anggota mendapatkan artikel berbeda.
selected = selected.sample(frac=1, random_state=SEED).reset_index(drop=True)
chunks = [selected.iloc[i::3].copy() for i in range(3)]
for i, chunk in enumerate(chunks, start=1):
    chunk['annotator'] = f'anggota_{i}'
    chunk.to_csv(OUT/f'annotation_queue_member_{i}.csv', index=False)
    print(f'Antrean anggota {i}: {len(chunk)} cuplikan')

display(selected[['article_ref','title','body_excerpt']].head(5))

## 5. Pedoman anotasi manual

Untuk setiap baris, isi `annotations_json` dengan array JSON. Beri label untuk setiap kemunculan entitas yang termasuk kategori yang disepakati kelompok. Contoh untuk teks `Ada Lovelace visited London.`:

```json
[
  {"text":"Ada Lovelace","label":"PERSON","start":0,"end":12},
  {"text":"London","label":"GPE","start":21,"end":27}
]
```

- `start` adalah indeks karakter awal, berbasis nol.
- `end` adalah indeks karakter akhir eksklusif, jadi `text_to_annotate[start:end]` harus sama persis dengan `text` entitas.
- Label minimal: `PERSON`, `ORG`, dan tempat (`GPE` atau `LOC`, sesuai pedoman kelompok). Agar konsisten, gunakan label yang sama untuk jenis yang sama.
- Tandai setiap kemunculan entitas jika muncul lebih dari sekali.
- Jika teks tidak memiliki entitas bernama, isi dengan `[]` (array kosong), bukan dibiarkan kosong.
- Jangan menganggap kolom `entities` sumber sebagai gold label tanpa validasi.

**Pemeriksaan wajib:** untuk setiap entitas, periksa bahwa `text_to_annotate[start:end] == text`. Notebook training nanti akan menolak anotasi yang offset-nya salah.

## 6. Unduh paket antrean anotasi

Bagikan `annotation_queue_member_1.csv` kepada anggota 1, file anggota 2 kepada anggota 2, dan seterusnya. Setelah diisi, ketiga file harus digabungkan untuk pemeriksaan silang. Jangan melakukan train/test split berdasarkan anggota; split dilakukan setelah anotasi selesai, berdasarkan `original_id`.

In [ ]:
guidelines = '''PEDOMAN ANOTASI NER\n\n1. Anotasi span-level pada text_to_annotate.\n2. Isi annotations_json dengan array JSON.\n3. Label minimal PERSON, ORG, GPE/LOC sesuai keputusan kelompok.\n4. start inklusif, end eksklusif, indeks karakter mulai dari 0.\n5. Pastikan text_to_annotate[start:end] persis sama dengan nilai text.\n6. Tandai setiap kemunculan entitas; bila tidak ada entitas, gunakan [].\n7. Jangan membuat label berdasarkan tebakan; diskusikan kasus ambigu.\n8. Catat pertanyaan atau koreksi di review_notes.\n'''
(OUT/'ANNOTATION_GUIDELINES.txt').write_text(guidelines, encoding='utf-8')
zip_out = shutil.make_archive('/content/NER_Annotation_Starter_Pack', 'zip', OUT)
print('Paket anotasi siap:', zip_out)
print('Isi folder:', sorted(p.name for p in OUT.iterdir()))
files.download(zip_out)

## 7. Apa yang dilakukan setelah ini?

1. Ketiga anggota mengisi antrean masing-masing.
2. Satukan menjadi satu CSV atau unggah ketiga CSV untuk digabungkan di notebook lanjutan.
3. Periksa konsistensi label dan seluruh offset.
4. Pastikan set evaluasi final mencakup minimal 10 artikel/cuplik-an berlabel, termasuk jika memungkinkan artikel tanpa entitas dan contoh yang menantang.
5. Tetapkan split 70% train, 15% validation, 15% test berdasarkan artikel setelah anotasi diperiksa. Set test harus tetap tidak tersentuh saat tuning.

Setelah berkas anotasi terkumpul, tahap selanjutnya adalah notebook training: baseline pretrained, fine-tuning/adaptasi, evaluasi exact-span, dan analisis kesalahan.